# Embedded AI Final Project

#### 20223265 조해영, 20214904 조성

0. 환경 설정

In [1]:
import json
import platform
import sys

import torch
import torchvision

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
else:
    gpu_name = None
    print("[warn] No GPU detected. Check Runtime > Change runtime type > GPU.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV_INFO = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "platform": platform.platform(),
}
print()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))

python: 3.13.9
torch: 2.14.0+cpu
torchvision: 0.29.0+cpu
cuda available: False
[warn] No GPU detected. Check Runtime > Change runtime type > GPU.

{
  "python_version": "3.13.9",
  "torch_version": "2.14.0+cpu",
  "torchvision_version": "0.29.0+cpu",
  "device": "cpu",
  "gpu_name": null,
  "platform": "Windows-11-10.0.26200-SP0"
}


### 1. 모델 호출

Classification을 위한 ViT를 호출한다

In [6]:
import timm
from timm.data import create_transform, resolve_model_data_config

MODEL_NAME = "vit_base_patch16_224"
model = timm.create_model(
    MODEL_NAME, 
    num_classes=1000,
    pretrained=True).to(DEVICE)
model.eval()

data_config = resolve_model_data_config(model)
eval_transform = create_transform(**data_config, is_training=False)

print("model:", MODEL_NAME)
print("device:", DEVICE)
print("preprocessing:", data_config)

model: vit_base_patch16_224
device: cpu
preprocessing: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


### ImageNet calibration / test 데이터

In [10]:
from pathlib import Path

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

# Place the extracted ImageNet-1k data under this folder:
# data/imagenet/train/<class_name>/*.JPEG
# data/imagenet/val/<class_name>/*.JPEG
IMAGENET_ROOT = Path("./data/imagenet")
IMAGENET_TRAIN_DIR = IMAGENET_ROOT / "train"
IMAGENET_VAL_DIR = IMAGENET_ROOT / "val"
CALIBRATION_SIZE = 100
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable in notebooks on Windows.
SEED = 42

missing_dirs = [
    str(path) for path in (IMAGENET_TRAIN_DIR, IMAGENET_VAL_DIR)
    if not path.is_dir()
]
if missing_dirs:
    raise FileNotFoundError(
        "ImageNet-1k train/val folders were not found: "
        + ", ".join(missing_dirs)
        + ". Download ImageNet-1k from its official source, extract it, and "
        "arrange both splits into class subfolders under data/imagenet/."
    )

train_dataset = ImageFolder(IMAGENET_TRAIN_DIR, transform=eval_transform)
val_dataset = ImageFolder(IMAGENET_VAL_DIR, transform=eval_transform)
if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError("Train and val folders must contain the same 1,000 class folders.")

if not 0 < CALIBRATION_SIZE <= len(train_dataset):
    raise ValueError(
        f"CALIBRATION_SIZE must be between 1 and {len(train_dataset)}, "
        f"got {CALIBRATION_SIZE}."
    )

rng = torch.Generator().manual_seed(SEED)
calibration_indices = torch.randperm(len(train_dataset), generator=rng)[:CALIBRATION_SIZE].tolist()
calibration_dataset = Subset(train_dataset, calibration_indices)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE == "cuda",
}
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

print(f"ImageNet classes: {len(train_dataset.classes)}")
print(f"Calibration images: {len(calibration_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Image transform: {eval_transform}")

FileNotFoundError: ImageNet-1k train/val folders were not found: data\imagenet\train, data\imagenet\val. Download ImageNet-1k from its official source, extract it, and arrange both splits into class subfolders under data/imagenet/.